In [ ]:
import pandas as pd


In [ ]:
df = pd.read_excel("online_retail_II.xlsx")

In [ ]:
df = df.dropna(subset=["StockCode","Customer ID"])
df = df[df["Quantity"] > 0]

In [ ]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [ ]:
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Month"] = df["InvoiceDate"].dt.month
df["Year"] = df["InvoiceDate"].dt.year

In [ ]:
daily_sales = df.groupby(
["StockCode","InvoiceDate"]
)["Quantity"].sum().reset_index()

In [ ]:
df.to_csv("online_retail_clean.csv", index=False)
daily_sales.to_csv("daily_sales.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_excel("online_retail_clean.xlsx")

# Cleaning
df = df.dropna(subset=["StockCode", "Customer ID"])
df = df[df["Quantity"] > 0]

# Date features
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["DayOfWeek"] = df["InvoiceDate"].dt.dayofweek
df["Month"] = df["InvoiceDate"].dt.month
df["Year"] = df["InvoiceDate"].dt.year

# Warehouse assignment rules
w1_countries = {
    'France', 'Germany', 'EIRE', 'Belgium', 'Netherlands',
    'Portugal', 'Denmark', 'Poland', 'Channel Islands',
    'Spain', 'Cyprus', 'Greece', 'Norway', 'Austria',
    'Sweden', 'Finland', 'Italy', 'Switzerland',
    'Iceland', 'Malta', 'Lithuania'
}

w2_countries = {
    'USA', 'Canada'
}

def assign_warehouse(country):
    if country in w1_countries:
        return 'W1'
    elif country in w2_countries:
        return 'W2'
    else:
        return 'W3'

df["WarehouseID"] = df["Country"].apply(assign_warehouse)

# Optional: aggregated daily sales for ML
daily_sales = df.groupby(["StockCode", "InvoiceDate"])["Quantity"].sum().reset_index()

# Save files
df.to_csv("online_retail_clean_with_warehouse.csv", index=False)
daily_sales.to_csv("daily_sales.csv", index=False)

print("Files created successfully!")
print(df["WarehouseID"].value_counts())
print(df[["Country", "WarehouseID"]].drop_duplicates().sort_values(by=["WarehouseID", "Country"]))

Files created successfully!
WarehouseID
W3    372910
W1     34478
W2       307
Name: count, dtype: int64
                     Country WarehouseID
12860                Austria          W1
173                  Belgium          W1
6977         Channel Islands          W1
8991                  Cyprus          W1
3749                 Denmark          W1
408                     EIRE          W1
19332                Finland          W1
71                    France          W1
543                  Germany          W1
11488                 Greece          W1
331849               Iceland          W1
25349                  Italy          W1
206760             Lithuania          W1
53239                  Malta          W1
4421             Netherlands          W1
12663                 Norway          W1
4486                  Poland          W1
1782                Portugal          W1
7947                   Spain          W1
14789                 Sweden          W1
26609            Switzerland      

In [ ]:
import pandas as pd

df = pd.read_excel("online_retail_II.xlsx")

# Cleaning
df = df.dropna(subset=["StockCode", "Customer ID"])
df = df[df["Quantity"] > 0]

# Convert date
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["InvoiceDateStr"] = df["InvoiceDate"].dt.strftime("%Y-%m-%dT%H:%M:%S")

# Clean ID fields
df["Customer ID"] = df["Customer ID"].astype(int).astype(str)
df["Invoice"] = df["Invoice"].astype(str)
df["StockCode"] = df["StockCode"].astype(str)

# Warehouse assignment
w1_countries = {
    'France', 'Germany', 'EIRE', 'Belgium', 'Netherlands',
    'Portugal', 'Denmark', 'Poland', 'Channel Islands',
    'Spain', 'Cyprus', 'Greece', 'Norway', 'Austria',
    'Sweden', 'Finland', 'Italy', 'Switzerland',
    'Iceland', 'Malta', 'Lithuania'
}

w2_countries = {'USA', 'Canada'}

def assign_warehouse(country):
    if country in w1_countries:
        return 'W1'
    elif country in w2_countries:
        return 'W2'
    else:
        return 'W3'

df["WarehouseID"] = df["Country"].apply(assign_warehouse)

# Base composite key
df["BaseTransactionKey"] = (
    df["WarehouseID"] + "#" +
    df["Invoice"] + "#" +
    df["Customer ID"] + "#" +
    df["InvoiceDateStr"]
)

# Add duplicate counter
df["LineNo"] = df.groupby(["StockCode", "BaseTransactionKey"]).cumcount()

# Final unique sort key
df["TransactionKey"] = (
    df["BaseTransactionKey"] + "#LINE" + df["LineNo"].astype(str)
)

# Verify uniqueness
duplicate_count = df.duplicated(subset=["StockCode", "TransactionKey"]).sum()
print("Duplicate key count:", duplicate_count)

# Save
df.to_csv("online_retail_transactions.csv", index=False)

print(df[["StockCode", "TransactionKey"]].head())

Duplicate key count: 0
  StockCode                             TransactionKey
0     85048  W3#489434#13085#2009-12-01T07:45:00#LINE0
1    79323P  W3#489434#13085#2009-12-01T07:45:00#LINE0
2    79323W  W3#489434#13085#2009-12-01T07:45:00#LINE0
3     22041  W3#489434#13085#2009-12-01T07:45:00#LINE0
4     21232  W3#489434#13085#2009-12-01T07:45:00#LINE0


In [ ]:
import pandas as pd

df = pd.read_csv("online_retail_transactions.csv")

print("Total rows in file:", len(df))

duplicate_keys = df.duplicated(subset=["StockCode", "TransactionKey"]).sum()
print("Duplicate key count:", duplicate_keys)

Total rows in file: 407695
Duplicate key count: 0


In [ ]:
import pandas as pd

# Load cleaned warehouse-aware transactional file
df = pd.read_csv("online_retail_clean_with_warehouse.csv")

# Ensure correct types
df["StockCode"] = df["StockCode"].astype(str)
df["WarehouseID"] = df["WarehouseID"].astype(str)
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")

# Drop rows with missing required fields
df = df.dropna(subset=["StockCode", "WarehouseID", "Quantity", "Price", "InvoiceDate"])

# Create inventory summary table
inventory_products = df.groupby(["StockCode", "WarehouseID"], as_index=False).agg({
    "Quantity": "sum",
    "Price": "mean",
    "InvoiceDate": "max",
    "Description": "first",
    "Country": "first"
})

# Rename columns
inventory_products.rename(columns={
    "Quantity": "CurrentStock",
    "Price": "AveragePrice",
    "InvoiceDate": "LastUpdated"
}, inplace=True)

# Format columns
inventory_products["AveragePrice"] = inventory_products["AveragePrice"].round(2)
inventory_products["LastUpdated"] = pd.to_datetime(
    inventory_products["LastUpdated"]
).dt.strftime("%Y-%m-%dT%H:%M:%S")

# Check duplicate keys
duplicate_count = inventory_products.duplicated(
    subset=["StockCode", "WarehouseID"]
).sum()

print("Duplicate key count:", duplicate_count)
print("Rows in inventory summary:", len(inventory_products))
print(inventory_products.head())

# Save output
inventory_products.to_csv("inventory_products.csv", index=False)
print("inventory_products.csv created successfully")

Duplicate key count: 0
Rows in inventory summary: 7032
  StockCode WarehouseID  CurrentStock  AveragePrice          LastUpdated  \
0     10002          W1          1784          0.84  2010-12-09T14:49:00   
1     10002          W3          6017          0.84  2010-12-09T18:58:00   
2     10080          W3            12          0.85  2010-11-21T11:28:00   
3     10109          W3             4          0.42  2009-12-03T12:31:00   
4     10120          W1            32          0.21  2010-11-23T14:06:00   

                   Description         Country  
0  INFLATABLE POLITICAL GLOBE            Spain  
1  INFLATABLE POLITICAL GLOBE   United Kingdom  
2     GROOVY CACTUS INFLATABLE  United Kingdom  
3         BENDY COLOUR PENCILS  United Kingdom  
4                 DOGGY RUBBER          France  
inventory_products.csv created successfully


In [ ]:
duplicate_count = inventory_products.duplicated(
    subset=["StockCode", "WarehouseID"]
).sum()

print("Duplicate key count:", duplicate_count)

Duplicate key count: 0


In [ ]:
import pandas as pd

df = pd.read_csv("inventory_products.csv")
print("Rows in CSV:", len(df))

Rows in CSV: 7032
